# Baseline Comparison - How do you know your ML model is actually better than a simple rule?


## The question

* Why is this an ML problem and not a lookup table of sector averages? *

Every ML project should be able to answer that **with a number**. A model is only worth its
complexity if it clearly beats the dumbest thing that could work. So we build a ladder of
lookup-table 'models' - no training, no features, just group medians - and score each on the
**same held-out test set** the real model was scored on.

## Rules that keep the comparison honest

1. **Same test set, same split.** We reproduce the training split inline (mirroring
   `create_train_val_test_split` in `src/model_building/mb_main.py`), so the rows are identical to
   training while keeping this notebook self-contained — it imports nothing from `src/`.

2. **Baselines are built from TRAINING data only.** Computing sector medians over the full dataset
   would let the baseline peek at test rows.

3. **Fallbacks for unseen groups.** A test sector missing from train falls back to a coarser
   median, never to a NaN.

## Why price-per-sqft matters

A plain *sector median price* baseline predicts the same value for a 800 sqft studio and a
4,000 sqft penthouse. It ignores `area` - the model's single strongest feature - so it makes the
model look good unfairly. The honest baselines multiply a median **price-per-sqft** by the
property's actual area.

## 1. Rebuild the exact test set

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import joblib
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score, mean_absolute_percentage_error
from sklearn.model_selection import train_test_split

In [19]:
Path.cwd()

WindowsPath('c:/Users/Jay Patel/Campusx/ml_projects/PropNavigator-Real-Estate-ML/notebooks/error_analysis')

In [ ]:
# Project root (self-contained: this notebook does NOT import from src/, so no
# stray logs and no heavy training imports).
ROOT = Path('C:/Users/Jay Patel/Campusx/ml_projects/PropNavigator-Real-Estate-ML')

df = pd.read_csv(ROOT / 'data/fs/feature_selected_properties.csv')

# Reproduce the training split inline (mirrors create_train_val_test_split in
# src/model_building/mb_main.py): stratified 60/20/20 on price quintiles, seed 42.
TARGET = 'price_in_cr'
X = df.drop(columns=[TARGET])
y = df[TARGET]
y_log = np.log1p(y)
bins = pd.qcut(y, q=5, labels=False)

X_temp, X_te, ytemp_log, yte_log, bins_temp, _ = train_test_split(
    X, y_log, bins, stratify=bins, test_size=0.2, random_state=42)
X_tr, X_val, ytr_log, yval_log = train_test_split(
    X_temp, ytemp_log, stratify=bins_temp, test_size=0.25, random_state=42)

In [9]:
# Convert log prices back to actual prices (Crores)
train = X_tr.copy()
train["price_in_cr"] = np.expm1(ytr_log).values

test = X_te.copy()
test["price_in_cr"] = np.expm1(yte_log).values

print(f"Train rows : {len(train)}")
print(f"Test rows  : {len(test)}")

Train rows : 22943
Test rows  : 7648


In [10]:
def mape(actual, pred):
    return mean_absolute_percentage_error(actual, pred) * 100

results = {}

## 2. The baseline ladder

Each rung adds one piece of information, so we can see what each is worth.

| Ladder              | How prediction is generated                                   |
| ------------------- | ------------------------------------------------------------- |
| Global Median       | Same training median for every test house                     |
| Global PPSF         | Training median PPSF × test house area                        |
| Sector Median       | Training sector median for that test house's sector           |
| Sector + PPSF       | Training sector median PPSF × test house area                 |
| Sector + Type       | Training sector & property type median PPSF × test house area |
| Sector + Type + BHK | Training sector + type + BHK median PPSF × test house area    |
| ML Model            | CatBoost predicts individually for each test house            |


In [ ]:
# ------------------------------------------------------------------
# Feature engineering
# ------------------------------------------------------------------

# ------------------------------------------------------------------
# RUNG 0 : Global Median Price
# ------------------------------------------------------------------

g_med = train["price_in_cr"].median()

pred = np.full(len(test), g_med)

results["Global Median Price"] = mape(test["price_in_cr"], pred)

# ------------------------------------------------------------------
# RUNG 0B : Global Median PPSF × Area
# ------------------------------------------------------------------

train["ppsf"] = train["price_in_cr"] / train["area"]

g_ppsf = train["ppsf"].median()

pred = g_ppsf * test["area"]

results["Global Median PPSF × Area"] = mape(test["price_in_cr"], pred)

# ------------------------------------------------------------------
# RUNG 1 : Sector Median Price
# ------------------------------------------------------------------

sector_price = train.groupby("sector")["price_in_cr"].median()

pred = (
    test["sector"]
    .map(sector_price)
    .fillna(g_med)
)

results["Sector Median Price"] = mape(test["price_in_cr"], pred)

# ------------------------------------------------------------------
# RUNG 2 : Sector Median PPSF × Area
# ------------------------------------------------------------------

sector_ppsf = train.groupby("sector")["ppsf"].median()

pred = (
    test["sector"]
    .map(sector_ppsf)
    .fillna(g_ppsf)
) * test["area"]

results["Sector Median PPSF × Area"] = mape(test["price_in_cr"], pred)

# ------------------------------------------------------------------
# RUNG 3 : Sector + Property Type
# ------------------------------------------------------------------

sector_type_ppsf = train.groupby(
    ["sector", "property_type"]
)["ppsf"].median()

pred = []

for sector, ptype in zip(test["sector"], test["property_type"]):

    value = sector_type_ppsf.get((sector, ptype))

    if pd.isna(value):
        value = sector_ppsf.get(sector, g_ppsf)

    pred.append(value)

pred = np.array(pred) * test["area"].values

results["Sector + Type PPSF × Area"] = mape(
    test["price_in_cr"],
    pred
)

# ------------------------------------------------------------------
# RUNG 4 : Sector + Type + Bedrooms
# ------------------------------------------------------------------

sector_type_bhk_ppsf = train.groupby(
    ["sector", "property_type", "bedRoom"]
)["ppsf"].median()

pred = []

for sector, ptype, bhk in zip(
    test["sector"],
    test["property_type"],
    test["bedRoom"]
):

    value = sector_type_bhk_ppsf.get((sector, ptype, bhk))

    if pd.isna(value):
        value = sector_type_ppsf.get((sector, ptype))

    if pd.isna(value):
        value = sector_ppsf.get(sector, g_ppsf)

    pred.append(value)

pred = np.array(pred) * test["area"].values

results["Sector + Type + BHK PPSF × Area"] = mape(
    test["price_in_cr"],
    pred
)

# ------------------------------------------------------------------
# ML MODEL
# ------------------------------------------------------------------

bundle = joblib.load(ROOT / "artifacts/best_model.joblib")

pred = np.expm1(
    bundle["pipeline"].predict(X_te)
)

results[f"ML MODEL ({bundle['model_name']})"] = mape(
    test["price_in_cr"],
    pred
)

# ------------------------------------------------------------------
# Results
# ------------------------------------------------------------------

table = (
    pd.DataFrame(
        {
            "Approach": results.keys(),
            "Test MAPE (%)": results.values()
        }
    )
    .sort_values("Test MAPE (%)", ascending=False)
    .round(2)
    .reset_index(drop=True)
)

print(table)

                          Approach  Test MAPE (%)
0              Global Median Price          72.79
1              Sector Median Price          48.29
2        Global Median PPSF × Area          33.18
3        Sector Median PPSF × Area          23.71
4        Sector + Type PPSF × Area          21.65
5  Sector + Type + BHK PPSF × Area          19.61
6              ML MODEL (LightGBM)          11.57


##### 1. Build lookup table
lookup = train.groupby(...).median()

###### 2. Generate predictions
pred = ...

###### 3. Evaluate
results["Name"] = mape(test["price_in_cr"], pred)

## 3. How much is the ML actually worth?

In [12]:
model_name = f"ML MODEL ({bundle['model_name']})"
model_mape = results[model_name]
best_lookup_name = min((k for k in results if k != model_name), key=lambda k: results[k])
best_lookup = results[best_lookup_name]

print(f'Best lookup table : {best_lookup:.2f}%   ({best_lookup_name})')
print(f'ML model          : {model_mape:.2f}%')
print(f'Absolute gain     : {best_lookup - model_mape:.2f} percentage points')
print(f'Relative gain     : {(1 - model_mape / best_lookup) * 100:.1f}% less error')
print()
print(f"Model R2 on test  : {r2_score(test['price_in_cr'], model_pred):.4f}")

out = ROOT / 'data/error_analysis'
out.mkdir(parents=True, exist_ok=True)
table.to_csv(out / 'baseline_comparison.csv', index=False)
print(f'\nSaved -> {out / "baseline_comparison.csv"}')

Best lookup table : 19.61%   (Sector + Type + BHK PPSF × Area)
ML model          : 11.57%
Absolute gain     : 8.04 percentage points
Relative gain     : 41.0% less error

Model R2 on test  : 0.9187

Saved -> C:\Users\Jay Patel\Campusx\ml_projects\PropNavigator-Real-Estate-ML\data\error_analysis\baseline_comparison.csv


## 4. Conclusion

| Approach | Test MAPE | What it knows |
|---|---|---|
| Global median price | **72.3%** | nothing |
| Sector median price (naive) | **47.8%** | location only |
| Global median Rs/sqft x area | **33.3%** | size only |
| Sector median Rs/sqft x area | **23.6%** | location + size |
| Sector x type median Rs/sqft x area | **21.4%** | + property type |
| Sector x type x BHK median Rs/sqft x area | **19.4%** | + bedrooms |
| **ML model (LightGBM)** | **11.4%** | all 24 features + interactions |

**The headline: the ML model beats the strongest lookup table by 8.0 points - a 41% reduction in
error.** That is a decisive answer to *"why not just use a spreadsheet?"*

### Two things worth noticing

**1. Size matters more than location.** Knowing only the property's area (33.3%) beats knowing
only its sector (47.8%). That independently corroborates the SHAP analysis, where `area` is the
dominant driver by a wide margin.

**2. The lookup ladder flattens out.** Adding property type buys 2.2 points; adding bedrooms buys
another 2.0. Extrapolating, no amount of extra grouping gets a lookup table near 11.4% - each new
split divides the data into ever-smaller, noisier groups. The remaining 8 points come from what a
lookup table structurally cannot do: model **non-linear interactions** (how age interacts with
sector, how the area premium changes across price bands) and use the continuous distance features.

